## ML_1046: INT8 Quantized Linear

**Difficulty**: Hard | **TorchCode**: #36

### Problem
Implement per-channel INT8 quantization for a linear layer. Quantize weights to int8, store as a buffer (not parameter). At forward time, dequantize and matmul in float32.

### Formula
$$s_i = \frac{\max|W_{i,\cdot}|}{127}, \quad W^{\text{int8}}_{i,j} = \text{clamp}\!\left(\text{round}\!\left(\frac{W_{i,j}}{s_i}\right), -128, 127\right)$$
$$y = x\,(W^{\text{int8}} \odot s)^T + b$$

In [ ]:
import torch
import torch.nn as nn

class Int8Linear(nn.Module):
    def __init__(self, weight, bias=None):
        super().__init__()
        scale = weight.abs().amax(dim=1, keepdim=True) / 127.0
        self.register_buffer('weight_int8',
            torch.round(weight / (scale + 1e-10)).clamp(-128, 127).to(torch.int8))
        self.register_buffer('scale', scale)
        self.bias = nn.Parameter(bias.clone()) if bias is not None else None

    def forward(self, x):
        w = self.weight_int8.float() * self.scale
        out = x @ w.T
        if self.bias is not None:
            out = out + self.bias
        return out

In [ ]:
import torch
import torch.nn as nn

# --- Test 1: Weight is int8 ---
w = torch.randn(32, 16)
q = Int8Linear(w)
assert isinstance(q, nn.Module)
assert q.weight_int8.dtype == torch.int8

# --- Test 2: Values in [-128, 127] ---
q2 = Int8Linear(torch.randn(64, 32) * 10)
assert q2.weight_int8.min() >= -128 and q2.weight_int8.max() <= 127

# --- Test 3: Dequantized close to original ---
torch.manual_seed(0)
w = torch.randn(16, 8)
q3 = Int8Linear(w)
w_recon = q3.weight_int8.float() * q3.scale
assert (w - w_recon).abs().max() < 0.1

# --- Test 4: Forward output shape ---
q4 = Int8Linear(torch.randn(8, 4), torch.randn(8))
assert q4(torch.randn(2, 4)).shape == (2, 8)

# --- Test 5: Weight is buffer not parameter ---
q5 = Int8Linear(torch.randn(4, 4))
param_names = [n for n, _ in q5.named_parameters()]
assert 'weight_int8' not in param_names
assert 'scale' not in param_names

print('All tests passed!')